In [1]:
import sys 
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd 
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from config import PROCESSED_DATA_DIR

In [2]:
pokemon_df = pd.read_parquet(PROCESSED_DATA_DIR / "pokemon_data_features.parquet")

## Baseline Dummy Model

In [3]:
y = pokemon_df["total_points"]
X = pokemon_df.drop(columns=["total_points"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Baseline RMSE: {rmse:.2f}")
print(f"Baseline MAE: {mae:.2f}")
print(f"Baseline R^2: {r2:.3f}")

Baseline RMSE: 124.00
Baseline MAE: 99.92
Baseline R^2: -0.003


## Linear Regression Model

In [4]:
numeric_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="passthrough"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Linear RMSE: {rmse}")
print(f"Linear MAE: {mae}")
print(f"Linear R^2: {r2}")

Linear RMSE: 68.22459919882877
Linear MAE: 55.45987567935343
Linear R^2: 0.69640245960052


## Ridge Regression Model

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols)
    ],
    remainder="passthrough"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=1.0))
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Linear RMSE: {rmse}")
print(f"Linear MAE: {mae}")
print(f"Linear R^2: {r2}")

Linear RMSE: 68.22374928177551
Linear MAE: 55.458345231450615
Linear R^2: 0.6964100237671903


## Lasso Regression Model

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols)
    ],
    remainder="passthrough"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Lasso(alpha=0.01, max_iter=10_000))
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Linear RMSE: {rmse}")
print(f"Linear MAE: {mae}")
print(f"Linear R^2: {r2}")

Linear RMSE: 68.21106728327796
Linear MAE: 55.44329433362976
Linear R^2: 0.6965228809413004
